In [2]:
import glob
import math
import pickle
import numpy as np
import pandas as pd

from rashomon import AIS, MCMC, hasse, loss
from rashomon.aggregate import RAggregate

DATA_DTA  = "../Dat/Does_Price_Matter_AER_merged.dta"
OUT_DIR   = "./output_charity_mcmc"
AIS_JSONL = f"{OUT_DIR}/charity_ais_300.jsonl"
PB_PKL    = f"{OUT_DIR}/charity_pb_300.pkl"   # optional

In [3]:
raw_df = pd.read_stata(DATA_DTA)
cols_to_keep = ["treatment", "control", "ratio", "size", "ask",
                "amount", "gave", "amountchange", "red0"]
df = raw_df[cols_to_keep].copy().dropna()

df["ratio"] = df["ratio"].map({"Control": 0, 1: 1, 2: 2, 3: 3})
df["size"]  = df["size"].map({"Control": 0, "$25,000": 1, "$50,000": 2, "$100,000": 3, "Unstated": 4})
df["ask"]   = df["ask"].map({"Control": 0, "1x": 1, "1.25x": 2, "1.50x": 3})
df["red0"]  = df["red0"].map({0: 1, 1: 2, np.nan: 0})
df = df.astype({"ratio": np.int64, "size": np.int64, "ask": np.int64, "red0": np.int64})
df = df.drop(["treatment", "control"], axis=1)

Z = df.to_numpy()
X = Z[:, [0, 1, 2, 6]]
y = (Z[:, 3] / 100).reshape(-1, 1)
num_data = X.shape[0]

M   = 4
R   = np.array([4, 5, 4, 3])
reg = 1e-7
q   = 0.0075579

all_policies = hasse.enumerate_policies(M, R)
num_policies = len(all_policies)
prof_idx_of_policy, profiles_hasse = AIS.build_profile_index_of_policy(
    all_policies, hasse.policy_to_profile
)

D = np.zeros((num_data, 1), dtype=np.int64)
for i in range(num_data):
    pol_i   = tuple(int(v) for v in X[i])
    D[i, 0] = next(j for j, p in enumerate(all_policies) if p == pol_i)

# Keep (0, 0) for no-data policies — do NOT set -inf sentinel.
# compute_pool_means guards on count==0; (sum=-inf, count=1) bypasses that guard
# and poisons every pool containing a no-data policy.
policy_means = loss.compute_policy_means(D, y, num_policies)

sigma2 = AIS.compute_sigma2_saturated(D, y, all_policies)
score_s = AIS.make_score_s_gprior(
    D=D, y=y, M=M, R=R,
    prof_idx_of_policy=prof_idx_of_policy,
    policies=all_policies, policy_means=policy_means,
    g=float(num_data), sigma2=sigma2, lam=reg,
)
normal_kw = dict(
    D=D, y=y, M=M,
    policies=all_policies,
    prof_idx_of_policy=prof_idx_of_policy,
    R=R, g=float(num_data), sigma2=sigma2,
    lattice_edges=None,
    p=[0.025, 0.5, 0.975],
)
print(f"{num_data} observations, {num_policies} policies")

50048 observations, 240 policies


In [4]:
# RPS via RAggregate
R_set, rashomon_profiles = RAggregate(M, R, np.inf, D, y, q, reg=reg)
RPS_states = AIS.raggregate_to_states((R_set, rashomon_profiles), profiles_hasse)
log_alpha  = [math.log(max(1e-300, score_s(s))) for s in RPS_states]

rps_logw  = np.asarray(log_alpha, float)
rps_normw = np.exp(rps_logw - np.max(rps_logw))
rps_normw /= rps_normw.sum()

rps_mean      = AIS.estimate_policy_means_from_RPS(
    RPS_states, rps_logw, all_policies, policy_means,
    prof_idx_of_policy, R, M, lattice_edges=None,
)
rps_quantiles = AIS.states_quantiles_normal_for_all_policies(RPS_states, rps_logw, **normal_kw)
rps_n         = len(RPS_states)
rps_ess       = float(1.0 / np.sum(rps_normw ** 2))
print(f"RPS: {rps_n} states, ESS={rps_ess:.1f}")

RPS: 123 states, ESS=123.0


In [5]:
# MCMC: load all chain jsonl files
chain_files = sorted(glob.glob(f"{OUT_DIR}/mcmc_chain_*.jsonl"))
print(f"Found {len(chain_files)} chain file(s): {chain_files}")

all_samples = []
for cf in chain_files:
    res = MCMC.load_mcmc_res_from_jsonl(cf)
    all_samples.extend(res["samples"])
    print(f"  {cf}: {len(res['samples'])} samples")

n_mcmc    = len(all_samples)
mcmc_logw = np.log(np.full(n_mcmc, 1.0 / n_mcmc))

mcmc_mean      = AIS.estimate_policy_means_from_RPS(
    all_samples, mcmc_logw, all_policies, policy_means,
    prof_idx_of_policy, R, M, lattice_edges=None,
)
mcmc_quantiles = AIS.states_quantiles_normal_for_all_policies(all_samples, mcmc_logw, **normal_kw)
mc_lo = np.asarray(mcmc_quantiles["0.025"])
mc_hi = np.asarray(mcmc_quantiles["0.975"])
print(f"MCMC total: {n_mcmc} samples")

Found 1 chain file(s): ['./output_charity_mcmc/mcmc_chain_0.jsonl']
  ./output_charity_mcmc/mcmc_chain_0.jsonl: 3000 samples
MCMC total: 3000 samples


In [6]:
# AIS: load from jsonl
_r        = AIS.load_ais_from_jsonl(AIS_JSONL)
ais_logw  = np.asarray(_r["logw"], float)
ais_normw = np.exp(ais_logw - np.max(ais_logw))
ais_normw /= ais_normw.sum()
ais_out   = AIS.AISOutput(terminals=_r["terminals"], logw=ais_logw, normw=ais_normw, ladder=None)

ais_mean      = AIS.estimate_policy_means_from_ais(
    ais_out=ais_out, all_policies=all_policies,
    policy_means=policy_means, prof_idx_of_policy=prof_idx_of_policy,
    lattice_edges=None, R_per=R, M=M,
)
ais_quantiles = AIS.ais_quantiles_normal_for_all_policies(ais_out, **normal_kw)
ais_n         = len(ais_out.terminals)
ais_ess       = float(1.0 / np.sum(ais_normw ** 2))
print(f"AIS: {ais_n} terminals, ESS={ais_ess:.1f}")

AIS: 300 terminals, ESS=97.2


In [7]:
# PB: load from pkl if available
import os
pb_row = None
if os.path.exists(PB_PKL):
    with open(PB_PKL, "rb") as f:
        pb_data = pickle.load(f)
    pb_states     = pb_data["pb_states"]
    pb_log_scores = np.asarray(pb_data["pb_log_scores"], float)
    pb_normw      = np.exp(pb_log_scores - np.max(pb_log_scores))
    pb_normw     /= pb_normw.sum()
    pb_out        = AIS.AISOutput(terminals=pb_states, logw=pb_log_scores, normw=pb_normw, ladder=None)
    pb_mean       = AIS.estimate_policy_means_from_RPS(
        pb_states, pb_log_scores, all_policies, policy_means,
        prof_idx_of_policy, R, M, lattice_edges=None,
    )
    pb_quantiles  = AIS.ais_quantiles_normal_for_all_policies(pb_out, **normal_kw)
    pb_n          = len(pb_states)
    pb_ess        = float(1.0 / np.sum(pb_normw ** 2))
    pb_row = dict(method="PB", n_states=pb_n, ESS=pb_ess, ESS_ratio=pb_ess/pb_n,
                  L1_mean=None, L1_q025=None, L1_q975=None, IoU=None,
                  runtime_s=pb_data.get("pb_time", float("nan")))
    print(f"PB: {pb_n} states, ESS={pb_ess:.1f}")
else:
    print(f"PB pkl not found ({PB_PKL}), skipping.")

PB: 423 states, ESS=423.0


In [9]:
def l1(a, b):
    return float(np.sum(np.abs(np.asarray(a) - np.asarray(b))))

def ci_iou(lo_m, hi_m, lo_ref, hi_ref):
    inter = np.maximum(0, np.minimum(hi_m, hi_ref) - np.maximum(lo_m, lo_ref))
    ref   = hi_ref - lo_ref
    return float(np.nanmean(np.where(ref > 0, inter / ref, np.nan)))

rows = [
    dict(method="RPS",
         n_states=rps_n, ESS=rps_ess, ESS_ratio=rps_ess/rps_n,
         L1_mean=l1(rps_mean, mcmc_mean),
         L1_q025=l1(rps_quantiles["0.025"], mc_lo),
         L1_q975=l1(rps_quantiles["0.975"], mc_hi),
         IoU=ci_iou(rps_quantiles["0.025"], rps_quantiles["0.975"], mc_lo, mc_hi),
         runtime_s=float("6.8")),
    dict(method="AIS",
         n_states=ais_n, ESS=ais_ess, ESS_ratio=ais_ess/ais_n,
         L1_mean=l1(ais_mean, mcmc_mean),
         L1_q025=l1(ais_quantiles["0.025"], mc_lo),
         L1_q975=l1(ais_quantiles["0.975"], mc_hi),
         IoU=ci_iou(ais_quantiles["0.025"], ais_quantiles["0.975"], mc_lo, mc_hi),
         runtime_s=float("1199.2")),
    dict(method="MCMC",
         n_states=n_mcmc, ESS=float(n_mcmc), ESS_ratio=1.0,
         L1_mean=0.0, L1_q025=0.0, L1_q975=0.0, IoU=1.0,
         runtime_s=float("2911.2")),
]

if pb_row is not None:
    pb_row.update(dict(
        L1_mean=l1(pb_mean, mcmc_mean),
        L1_q025=l1(pb_quantiles["0.025"], mc_lo),
        L1_q975=l1(pb_quantiles["0.975"], mc_hi),
        IoU=ci_iou(pb_quantiles["0.025"], pb_quantiles["0.975"], mc_lo, mc_hi),
    ))
    rows.insert(2, pb_row)

summary = pd.DataFrame(rows).set_index("method").round(4)
summary

,n_states,ESS,ESS_ratio,L1_mean,L1_q025,L1_q975,IoU,runtime_s
method,,,,,,,,
RPS,123,123.0000,1.000,0.0498,0.0757,0.0737,0.9970,6.8000
AIS,300,97.1997,0.324,0.0102,0.0146,0.0195,0.9912,1199.2000
PB,423,423.0000,1.000,0.0073,0.0135,0.0144,0.9999,1373.8158
MCMC,3000,3000.0000,1.000,0.0000,0.0000,0.0000,1.0000,2911.2000
